# Solutions — Context Managers

Exercise notebook: `05_functions_and_oop/05_context_managers.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

## Exercise 1 — 🟢 Easy · A timing context manager

**Working solution**

In [ ]:
import time

class Timer:
    def __init__(self, label="block"):
        self.label = label
        self.elapsed = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self._start
        return False

**Explanation**

`__enter__`'s return value is what `as` binds. Returning a falsy value from `__exit__` lets exceptions propagate; returning `True` would swallow them.

**Complexity:** O(1).

**Common mistakes**

- Returning `True` from `__exit__` by accident — silently hides errors.

## Exercise 2 — 🟡 Medium · Temporarily change the working directory

**Working solution**

In [ ]:
import os
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def working_directory(path):
    previous = Path.cwd()
    target = Path(path)
    target.mkdir(parents=True, exist_ok=True)
    os.chdir(target)
    try:
        yield target
    finally:
        os.chdir(previous)

**Explanation**

With `@contextmanager`, code before `yield` is `__enter__`, after is `__exit__`. The `try/finally` around `yield` is what guarantees restoration. (Python 3.11 ships `contextlib.chdir`.)

**Complexity:** O(1).

**Common mistakes**

- No `try/finally` around `yield` → the directory isn't restored when the block raises.

## Exercise 3 — 🟡 Medium · Temporary environment variables

**Working solution**

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def temp_env(**overrides):
    _missing = object()
    previous = {k: os.environ.get(k, _missing) for k in overrides}
    try:
        for key, value in overrides.items():
            if value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = str(value)
        yield
    finally:
        for key, old in previous.items():
            if old is _missing:
                os.environ.pop(key, None)
            else:
                os.environ[key] = old

**Explanation**

Snapshot first, mutate inside `try`, restore in `finally`. The sentinel distinguishes 'was unset' from 'was empty string'. (In pytest you'd use the `monkeypatch` fixture — which is implemented exactly like this.)

**Complexity:** O(k).

**Common mistakes**

- Restoring a previously-unset variable to `''` instead of deleting it.

## Exercise 4 — 🟠 Hard · Transactions with commit and rollback

**Working solution**

In [ ]:
from contextlib import contextmanager

_DELETED = object()

class KeyValueStore:
    def __init__(self):
        self._data = {}
        self._staged = None
        self.commits = 0
        self.rollbacks = 0

    def get(self, key, default=None):
        if self._staged is not None and key in self._staged:
            value = self._staged[key]
            return default if value is _DELETED else value
        return self._data.get(key, default)

    def set(self, key, value):
        if self._staged is not None:
            self._staged[key] = value
        else:
            self._data[key] = value

    def delete(self, key):
        if self._staged is not None:
            self._staged[key] = _DELETED
        else:
            self._data.pop(key, None)

    def committed(self):
        return dict(self._data)

    @contextmanager
    def transaction(self):
        if self._staged is not None:
            raise RuntimeError("nested transactions are not supported")
        self._staged = {}
        try:
            yield self
        except BaseException:
            self.rollbacks += 1
            raise
        else:
            for key, value in self._staged.items():
                if value is _DELETED:
                    self._data.pop(key, None)
                else:
                    self._data[key] = value
            self.commits += 1
        finally:
            self._staged = None

**Explanation**

`try/except/else/finally` around `yield` maps one-to-one onto rollback/commit/cleanup. This is the shape of SQLAlchemy's `session.begin()`
and of "write to staging, then publish" in pipelines.

**Complexity:** O(staged keys) per commit.

**Common mistakes**

- Applying writes directly and trying to 'undo' on failure.
- Suppressing the exception after rollback.

## Challenge — 🔴 Very Hard · Many files at once with ExitStack: k-way merge of sorted files

**Working solution**

In [ ]:
import csv
import heapq
from contextlib import ExitStack
from pathlib import Path

def merge_sorted_files(paths, output_path, key_column):
    if not paths:
        return 0
    with ExitStack() as stack:
        readers = []
        for path in paths:
            f = stack.enter_context(open(path, newline="", encoding="utf-8"))
            readers.append(csv.DictReader(f))
        headers = [r.fieldnames for r in readers]
        if any(h != headers[0] for h in headers):
            raise ValueError(f"headers differ: {headers}")
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        out = stack.enter_context(open(output_path, "w", newline="", encoding="utf-8"))
        writer = csv.DictWriter(out, fieldnames=headers[0])
        writer.writeheader()
        count = 0
        for row in heapq.merge(*readers, key=lambda r: r[key_column]):
            writer.writerow(row)
            count += 1
        return count

**Explanation**

`ExitStack` is a dynamic `with`: everything entered is closed in reverse order on exit or error. Combined with lazy `DictReader`s and
`heapq.merge`, memory is O(number of files), regardless of file sizes — the merge phase of an external sort (and of Spark's sort-merge join).

**Complexity:** O(N log k) time, O(k) memory.

**Common mistakes**

- Opening files in a list comprehension without ExitStack → leaked handles on error.
- Reading every file into a list, concatenating and sorting.